# Phase 7 - Customer Risk Scoring & Power BI Preparation (Real Execution)

Uses the **saved Phase-6 pipeline** (`models/churn_model.pkl`) exactly as-is - no retraining.
All logic lives in `src/risk_scoring.py`; this notebook calls the same functions the
artifacts are written from.

Three distinct concepts are kept separate throughout:

| Concept | Column | Meaning |
|---|---|---|
| Model probability | `churn_probability` | Continuous score from `predict_proba[:, 1]` |
| Classification flag | `predicted_churn_at_0.40` | Binary: probability >= 0.40 |
| Business risk segment | `risk_segment` | Low <0.25 / Medium 0.25-<0.50 / High 0.50-<0.75 / Critical >=0.75 |

In [1]:
import sys
from pathlib import Path

import pandas as pd

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import src.preprocessing as prep
import src.risk_scoring as rs

pd.set_option("display.width", 200)
print("Model file :", rs.MODEL_PATH)
print("Source data:", prep.MODEL_READY_PATH)

Model file : models\churn_model.pkl
Source data: data\processed\model_ready_data.csv


## 1. Load the saved model (verify preprocessing + classifier exist)

In [2]:
model = rs.load_and_verify_model()  # asserts steps == [preprocess, classifier] and LogisticRegression
print("Pipeline steps  :", list(model.named_steps.keys()))
print("Classifier      :", model.named_steps["classifier"])
print("Preprocessor    :", type(model.named_steps["preprocess"]).__name__)

Pipeline steps  : ['preprocess', 'classifier']
Classifier      : LogisticRegression(class_weight='balanced', max_iter=1000)
Preprocessor    : ColumnTransformer


## 2. Load model-ready data

In [3]:
clean_sha_before = prep.clean_sha256()
df = prep.load_model_ready()
print(f"Loaded {df.shape[0]} rows x {df.shape[1]} cols | actual churn rate {df['Churn'].mean():.4f}")

Loaded 7043 rows x 26 cols | actual churn rate 0.2654


## 3. Score all 7,043 customers

`churn_probability` (continuous) and `predicted_churn_at_0.40` (binary flag) are separate columns.

In [4]:
scores = rs.score_customers(df, model)  # validates: 7043, unique IDs, prob in [0,1], no missing
print(f"Scored {len(scores)} customers")
print(f"churn_probability: min={scores['churn_probability'].min():.6f}, "
      f"max={scores['churn_probability'].max():.6f}, mean={scores['churn_probability'].mean():.4f}")
print(f"Flagged at 0.40  : {int(scores['predicted_churn_at_0.40'].sum())} "
      f"({scores['predicted_churn_at_0.40'].mean() * 100:.2f}%)")
scores.head()

Scored 7043 customers
churn_probability: min=0.011974, max=0.964460, mean=0.4124
Flagged at 0.40  : 3465 (49.20%)


,customerID,Churn,churn_probability,predicted_churn_at_0.40,risk_segment
0,7590-VHVEG,0,0.833967,1,Critical
1,5575-GNVDE,0,0.096450,0,Low
2,3668-QPYBK,1,0.569165,1,High
3,7795-CFOCW,0,0.107300,0,Low
4,9237-HQITU,1,0.889092,1,Critical


## 4. Risk segments

Rules: Low `<0.25` | Medium `0.25-<0.50` | High `0.50-<0.75` | Critical `>=0.75`.

In [5]:
seg_table, seg_checks = rs.segment_analysis(scores)
display(seg_table)
print("Totals check:", seg_checks)
print(f"Sum customers = {int(seg_table['customers'].sum())} (== 7043), "
      f"sum churned = {int(seg_table['actual_churned'].sum())} (== 1869)")

,risk_segment,customers,avg_predicted_risk,actual_churned,pct_customers,actual_churn_rate_pct
0,Low,2790,0.0991,109,39.6138,3.9068
1,Medium,1399,0.3787,259,19.8637,18.5132
2,High,1489,0.6303,568,21.1416,38.1464
3,Critical,1365,0.8498,933,19.3809,68.3516


Totals check: {'customers_sum_to_total': True, 'churned_sum_to_total': True, 'all_segments_nonempty': True}
Sum customers = 7043 (== 7043), sum churned = 1869 (== 1869)


## 5. Save predictions (scores + high-risk export)

In [6]:
s_path, h_path = rs.save_predictions(scores)
high = scores[scores["risk_segment"].isin(rs.HIGH_RISK_SEGMENTS)].sort_values("churn_probability", ascending=False)
print(f"Saved {s_path}  ({len(scores)} rows)")
print(f"Saved {h_path}  ({len(high)} rows = High + Critical, sorted by probability desc)")
high.head(5)

Saved outputs\predictions\customer_risk_scores.csv  (7043 rows)
Saved outputs\predictions\high_risk_customers.csv  (2854 rows = High + Critical, sorted by probability desc)


,customerID,Churn,churn_probability,predicted_churn_at_0.40,risk_segment
3380,5178-LMXOP,1,0.964460,1,Critical
1976,9497-QCMMS,1,0.963933,1,Critical
4800,9300-AGZNL,1,0.963706,1,Critical
3749,4424-TKOPW,1,0.963531,1,Critical
3159,5150-ITWWB,0,0.963411,1,Critical


## 6. Business segment analysis

Customer count / average predicted risk / actual historical churn rate per segment
(observed differences only - no causal claims).

In [7]:
biz = rs.business_segment_analysis(df, scores)
for name, t in biz.items():
    print(f"=== {name} ===")
    display(t)

=== Contract ===


,Contract,customers,avg_predicted_risk,actual_churned,actual_churn_rate_pct
0,Month-to-month,3875,0.6262,1655,42.7097
1,One year,1473,0.2390,166,11.2695
2,Two year,1695,0.0745,48,2.8319


=== TenureGroup ===


,TenureGroup,customers,avg_predicted_risk,actual_churned,actual_churn_rate_pct
0,≤6 months,1481,0.7069,784,52.9372
1,7–12 months,705,0.5448,253,35.8865
2,13–24 months,1024,0.4620,294,28.7109
3,25+ months,3833,0.2611,538,14.0360


=== InternetService ===


,InternetService,customers,avg_predicted_risk,actual_churned,actual_churn_rate_pct
0,Fiber optic,3096,0.6043,1297,41.8928
1,DSL,2421,0.3292,459,18.9591
2,No,1526,0.1553,113,7.4050


... (6 outputs left out)

=== 0–6 months + Fiber optic ===


,customers,avg_predicted_risk,actual_churned,actual_churn_rate_pct
0,620,0.8888,460,74.1935


=== Month-to-month + 0–6 months ===


,customers,avg_predicted_risk,actual_churned,actual_churn_rate_pct
0,1413,0.7289,780,55.2017


## 7. Power BI dataset + data dictionary

In [8]:
pbi = rs.build_powerbi_dataset(df, scores)
pbi_paths = rs.save_powerbi(pbi)
print(f"powerbi_dataset.csv : {pbi.shape[0]} rows x {pbi.shape[1]} cols")
for p in pbi_paths:
    print("Saved:", p)
print("\nColumns:", list(pbi.columns))
display(pd.read_csv(rs.PBI_DICTIONARY_PATH).head(8))

powerbi_dataset.csv : 7043 rows x 29 cols
Saved: powerbi\powerbi_dataset.csv
Saved: powerbi\data_dictionary.csv
Saved: powerbi\dax_measures.md
Saved: powerbi\dashboard_documentation.md

Columns: ['customerID', 'gender', 'SeniorCitizen', 'Partner', 'Dependents', 'tenure', 'TenureGroup', 'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'ServiceCount', 'HasSecuritySupport', 'Contract', 'PaperlessBilling', 'PaymentMethod', 'MonthlyCharges', 'MonthlyChargeBand', 'TotalCharges', 'AvgMonthlyCharge', 'Churn', 'churn_probability', 'predicted_churn_at_0.40', 'risk_segment']


,Column,Data Type,Source,Description
0,customerID,str,Identifier,Unique customer identifier (join key for Power...
1,gender,str,Original,Customer gender (Female / Male)
2,SeniorCitizen,int64,Original,"1 if customer is a senior citizen (65+), else 0"
3,Partner,str,Original,Customer has a spouse or partner (Yes / No)
4,Dependents,str,Original,Customer has dependents such as children (Yes ...
5,tenure,int64,Original,Number of months the customer has been with th...
6,TenureGroup,str,Engineered,Engineered: relationship-age band derived from...
7,PhoneService,str,Original,Subscribes to home phone service (Yes / No)


## 8. DAX measures + dashboard documentation (3 pages)

In [9]:
dax_text = rs.PBI_DAX_PATH.read_text(encoding="utf-8")
doc_text = rs.PBI_DOC_PATH.read_text(encoding="utf-8")
print(f"dax_measures.md: {len(dax_text)} chars, {dax_text.count('```dax')} dax block")
print("Measures defined:", dax_text.count("= ") - dax_text.count("=="))
print(f"\ndashboard_documentation.md: {len(doc_text)} chars")
print("Pages:", [ln for ln in doc_text.splitlines() if ln.startswith("## Page")])

dax_measures.md: 2684 chars, 1 dax block
Measures defined: 11

dashboard_documentation.md: 4080 chars
Pages: ['## Page 1 - Executive Overview', '## Page 2 - Churn Drivers', '## Page 3 - Customer Risk']


## 9. Validation + risk report

In [10]:
validation = rs.validate_outputs(df, scores, pbi, model)
for r in validation:
    print(r)
report = rs.write_risk_report(df, scores, seg_table, seg_checks, biz, validation)
print("\nReport written:", report, "exists =", report.exists())

PASS  exactly 7043 customers scored  (7043)
PASS  customerID unique
PASS  probability within [0,1]  ([0.011974, 0.964460])
PASS  no missing risk scores
PASS  flag column in {0,1}
PASS  segments in {Low, Medium, High, Critical}
PASS  High+Critical count == high-risk export rows  (2854 == 2854)
PASS  high-risk export sorted by probability desc
PASS  high-risk export segments only High/Critical
PASS  Power BI dataset row count matches  (7043)
PASS  Power BI dataset matches risk-score dataset
PASS  saved model reproduces scores
PASS  raw dataset unchanged
PASS  cleaned dataset unchanged
PASS  model-ready dataset unchanged

Report written: outputs\reports\customer_risk_report.txt exists = True


## 10. Source integrity + conclusions

In [11]:
assert prep.clean_sha256() == clean_sha_before, "Cleaned dataset was modified!"
print("Cleaned SHA-256 unchanged:", clean_sha_before)
print("Raw SHA-256 unchanged    :", prep.raw_sha256())
print(f"""
PHASE 7 CONCLUSIONS (actual results)
------------------------------------
- 7043 customers scored with the saved Phase-6 pipeline (no retraining).
- Risk segments: Low 2790 (3.91% actual churn) | Medium 1399 (18.51%) |
  High 1489 (38.15%) | Critical 1365 (68.35%). Monotonic: higher predicted risk
  corresponds to higher observed historical churn.
- Flagged at 0.40: flagged pool trades precision for recall (OOF recall 0.8676);
  final business threshold needs cost/benefit analysis.
- High + Critical export: 2854 customers sorted by probability (retention contact list).
- Power BI package ready: dataset (7043 x 29), data dictionary, DAX measures,
  3-page dashboard documentation.
- Caveat: training-set customers are scored in-sample; scores are triage priorities,
  not calibrated probabilities. Associations are not causal effects.
""")
print("PHASE 7 RISK SCORING: SUCCESS")

Cleaned SHA-256 unchanged: 38e6f2dd278033a4617f880e8f11304c5fc6dce2c05ecef3f2a8bf48eeaa8996
Raw SHA-256 unchanged    : 16320c9c1ec72448db59aa0a26a0b95401046bef5d02fd3aeb906448e3055e91

PHASE 7 CONCLUSIONS (actual results)
------------------------------------
- 7043 customers scored with the saved Phase-6 pipeline (no retraining).
- Risk segments: Low 2790 (3.91% actual churn) | Medium 1399 (18.51%) |
  High 1489 (38.15%) | Critical 1365 (68.35%). Monotonic: higher predicted risk
  corresponds to higher observed historical churn.
- Flagged at 0.40: flagged pool trades precision for recall (OOF recall 0.8676);
  final business threshold needs cost/benefit analysis.
- High + Critical export: 2854 customers sorted by probability (retention contact list).
- Power BI package ready: dataset (7043 x 29), data dictionary, DAX measures,
  3-page dashboard documentation.
- Caveat: training-set customers are scored in-sample; scores are triage priorities,
  not calibrated probabilities. Associatio